# Simstrat inputs v2

This notebook is the proposed replacement for `Simstrat_inputs.ipynb`. It keeps intentional writes to live model files, but moves parsing, validation, precision, and file replacement into `kalden.core.simstrat`.

Safety invariants:

- Flow components are aligned before summation; incomplete rows never become partial totals.
- Unit conversion is not rounded to whole mmol/m3 values.
- Input values use significant-digit formatting rather than a fixed two decimals.
- A validated temporary file atomically replaces the requested live model file.


In [ ]:
from pathlib import Path

import pandas as pd

from kalden.core.datascience.pandas import df_smart_resample
from kalden.core.simstrat import (
    SimstratConfig,
    generate_inflow_content,
    mg_l_to_mmol_m3,
    sum_complete_flows,
    write_inflow_file,
)


## Configuration

Edit the setup paths once. `RUN_LIVE_PIPELINE` is the single execution guard; live writing remains the intended behavior when it is enabled.


In [ ]:
MODEL_SETUPS = {
    # Replace these three placeholders with the selected production setups.
    "schiffenen_amont": Path(r"P:\Lausanne\CHA10063\03 Projet\03 Etudes de projet\02 Calculs\02_Simstrat\03_model_setups\Schiffenen_amont\SELECT_SETUP.par"),
    "schiffenen_aval": Path(r"P:\Lausanne\CHA10063\03 Projet\03 Etudes de projet\02 Calculs\02_Simstrat\03_model_setups\Schiffenen_aval\SELECT_SETUP.par"),
    "morat": Path(r"P:\Lausanne\CHA10063\03 Projet\03 Etudes de projet\02 Calculs\02_Simstrat\03_model_setups\Morat\SELECT_SETUP.par"),
}

RUN_LIVE_PIPELINE = False
INPUT_FREQUENCY = "h"
VALUE_FORMAT = ".10g"
MOLAR_MASS_DIOXYGEN = 32.0


## Model loading

Loading is strict by default. A malformed or missing input stops the pipeline instead of leaving a partially loaded model that looks usable.


In [ ]:
def load_models(setups: dict[str, Path]) -> dict[str, SimstratConfig]:
    models = {}
    for name, setup_path in setups.items():
        model = SimstratConfig(setup_path)
        model.load_inputs(errors="raise", validate=True)
        models[name] = model
    return models


## Hydrology preparation

Resample each component onto the model frequency before summation. `sum_complete_flows` requires every selected component at every timestamp, fixing the previous `sum(axis=1)` behavior that treated missing tributaries as zero.


In [ ]:
def prepare_total_flow(
    components: dict[str, pd.Series],
    *,
    frequency: str = INPUT_FREQUENCY,
    start: pd.Timestamp | None = None,
    end: pd.Timestamp | None = None,
) -> pd.DataFrame:
    if not components:
        raise ValueError("At least one flow component is required.")

    aligned = []
    for name, series in components.items():
        frame = series.rename(name).to_frame().sort_index()
        frame = df_smart_resample(frame, frequency)
        aligned.append(frame)

    result = pd.concat(aligned, axis=1, join="outer").sort_index()
    if start is not None:
        result = result.loc[result.index >= pd.Timestamp(start)]
    if end is not None:
        result = result.loc[result.index <= pd.Timestamp(end)]

    result["Total"] = sum_complete_flows(
        result,
        columns=list(components),
        name="Total",
        missing="raise",
    )
    return result


## Oxygen conversion

Keep calculation precision in the dataframe and writer. Round only a display copy if a table needs fewer decimals.


In [ ]:
def oxygen_mg_l_to_model_units(values: pd.Series) -> pd.Series:
    converted = mg_l_to_mmol_m3(values, MOLAR_MASS_DIOXYGEN)
    converted.name = "oxygen_mmol_m3"
    return converted

# Regression example: this remains 4.84375, not 5.
assert abs(oxygen_mg_l_to_model_units(pd.Series([0.155])).iloc[0] - 4.84375) < 1e-12


## Validated live writer

The target is still the live file from the selected setup. The atomic option only prevents truncation if validation or writing fails.


In [ ]:
def write_model_input(
    model: SimstratConfig,
    input_key: str,
    frame: pd.DataFrame,
    deep_flows: list[dict],
    surface_flows: list[dict],
    *,
    headers: str = "",
) -> Path:
    relative_path = model.config["Input"][input_key]
    live_path = model.root_path / relative_path
    return write_inflow_file(
        live_path,
        frame,
        model.reference_date,
        deep_flows,
        surface_flows,
        headers=headers,
        value_format=VALUE_FORMAT,
        allow_missing=False,
        atomic=True,
    )

def write_fabm_input(
    model: SimstratConfig,
    filename: str,
    frame: pd.DataFrame,
    deep_flows: list[dict],
    surface_flows: list[dict],
    *,
    headers: str = "",
) -> Path:
    fabm = model.config["FABMConfig"]
    relative_dir = fabm.get("FABMInflowPath", fabm.get("PathFABMinflow"))
    if not relative_dir:
        raise KeyError("FABMConfig must define FABMInflowPath or PathFABMinflow.")
    return write_inflow_file(
        model.root_path / relative_dir / filename,
        frame,
        model.reference_date,
        deep_flows,
        surface_flows,
        headers=headers,
        value_format=VALUE_FORMAT,
        allow_missing=False,
        atomic=True,
    )


## Dry-run format check

This cell is independent of the network drive and can always run after the imports.


In [ ]:
demo = pd.DataFrame(
    {"oxygen": [0.00456789123, 0.00567891234]},
    index=pd.date_range("2024-01-01", periods=2, freq="h"),
)
demo_content = generate_inflow_content(
    demo,
    "2024-01-01",
    deep_flows=[{"depth": -2, "col": "oxygen", "header": "O2 [mmol/m3]"}],
    surface_flows=[],
    value_format=VALUE_FORMAT,
)
assert "0.00456789123" in demo_content
print(demo_content)


## Production orchestration

Port each lake-specific source loader from the old notebook into a named function, then call the helpers above. Avoid shared globals between lake blocks.


In [ ]:
if RUN_LIVE_PIPELINE:
    models = load_models(MODEL_SETUPS)
    # Example sequence for each model/input:
    # 1. Read source series into a dict[str, Series].
    # 2. Call prepare_total_flow(..., start=model.start_date, end=model.end_date).
    # 3. Build explicit deep_flows/surface_flows definitions.
    # 4. Call write_model_input or write_fabm_input.
    # 5. Reload with model.load_inputs(errors="raise") as a round-trip check.
    raise NotImplementedError(
        "Configure the selected setup paths and lake-specific source loaders first."
    )


## Migration map

- Keep source-data readers grouped by lake and input type.
- Replace every row-wise hydrology `sum(axis=1)` with `sum_complete_flows`.
- Replace `round(mgL_to_mmolm3(...))` with `mg_l_to_mmol_m3(...)`.
- Replace `data.inputs_generate_content` plus `open(..., 'w')` with `write_inflow_file`.
- Keep plotting and exploratory calibration cells in separate analysis notebooks.
